# 00 · Colab setup

Run this at the start of every Colab session. It:
1. mounts Google Drive (data and MLflow results live there, so they survive disconnects),
2. clones or updates the repo,
3. installs the package,
4. runs the setup check, which logs one MLflow run.

**Private repo?** Add a Colab secret (key icon on the left) named `GITHUB_TOKEN` with a
fine-grained GitHub token that has read access to this repo, and allow this notebook to use it.

In [ ]:
# --- settings -------------------------------------------------------------
REPO = "Kevin-ai-jpg/BachelorThesis-MachineLearning-RoadCycling"
BRANCH = "main"
DRIVE_DATA = "/content/drive/MyDrive/thesis-cycling/data"   # data + mlruns on Drive
EXTRAS = "track"            # add ",ml" for LightGBM/Optuna/SHAP experiments

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, pathlib
pathlib.Path(DRIVE_DATA).mkdir(parents=True, exist_ok=True)
os.environ["CYCLINGML_DATA"] = DRIVE_DATA
os.environ["CYCLINGML_MLRUNS"] = f"{DRIVE_DATA}/mlruns"
print("Data folder:", DRIVE_DATA)

In [ ]:
import os, subprocess
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # public repo or no secret set

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
repo_dir = "/content/repo"
if os.path.isdir(f"{repo_dir}/.git"):
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", repo_dir, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", repo_dir, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, repo_dir], check=True)
%cd /content/repo
!git log -1 --oneline

In [ ]:
!pip install -q -e ".[{EXTRAS}]"


In [ ]:
# Setup check: should print "Logged run <id>"
!python -m cyclingml.smoke configs/smoke.yaml

In [ ]:
# Optional: list logged runs
import mlflow
from cyclingml.tracking import tracking_uri
mlflow.set_tracking_uri(tracking_uri())
mlflow.search_runs(search_all_experiments=True)[["experiment_id", "run_id", "status", "metrics.random_score", "tags.git_commit"]]

**Done when:** the setup check prints a run id and the table above shows it.
Send the printed output to Claude if anything fails.